# File Write And Load Loops

This notebook exercises the normal file loop:

1. create an eager frame,
2. write `.h5` and `.fil`,
3. reload eagerly through `Frame(waterfall=...)`,
4. open file-backed for bounded reads,
5. inject into a safe copy with `Frame.open_copy(...)`.

The contract is that saved/reloaded data match, copy-backed
mutation leaves the source unchanged, and plots/readbacks see the
data that were actually written to disk.

In [ ]:
%matplotlib inline

from pathlib import Path

from IPython import get_ipython
from IPython.display import display
import matplotlib
_ipython = get_ipython()
if _ipython is not None:
    _ipython.run_line_magic("matplotlib", "inline")
    matplotlib.use("module://matplotlib_inline.backend_inline", force=True)

import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u

import setigen as stg

OUT = Path("generated")
OUT.mkdir(exist_ok=True)

np.set_printoptions(precision=4, suppress=True)
print("matplotlib backend:", matplotlib.get_backend())
print("setigen from:", stg.__file__)

In [ ]:
frame = stg.Frame(
    tchans=16,
    fchans=512,
    df=3 * u.Hz,
    dt=2 * u.s,
    fch1=6_000_001_533 * u.Hz,
    ascending=False,
    seed=21,
    source_name="Write-load loop",
)
frame.add_noise(8, noise_type="chi2")
frame.add_constant_signal(
    f_start=frame.get_frequency(frame.fchans // 2),
    drift_rate=0.5 * frame.unit_drift_rate,
    level=5,
    width=4 * frame.df,
    f_profile_type="sinc2",
    doppler_smearing=True,
)

h5_path = OUT / "loop_source.h5"
fil_path = OUT / "loop_source.fil"
frame.save_hdf5(h5_path)
frame.save_fil(fil_path)
h5_path, fil_path

In [ ]:
eager_h5 = stg.Frame(waterfall=h5_path)
eager_fil = stg.Frame(waterfall=fil_path)

print("h5 matches:", np.allclose(eager_h5.data, frame.data))
print("fil matches:", np.allclose(eager_fil.data, frame.data))
print("h5 source:", eager_h5.source_name)
print("fil source:", eager_fil.source_name)

In [ ]:
with stg.Frame.open(h5_path, mode="r") as backed:
    region = backed.read_frame(
        f_index_range=(220, 300),
        t_index_range=(0, backed.tchans),
    )
    print("file-backed?", backed.is_file_backed)
    print("region shape:", region.shape)
    print("region derived metadata:", region.metadata["derived"])

    fig, ax = plt.subplots(figsize=(8, 3))
    backed.plot(
        f_index_range=(220, 300),
        t_index_range=(0, backed.tchans),
        db=False,
        colorbar=True,
    )
    ax.set_title("Region read from file-backed HDF5")
    display(fig)
    plt.close(fig)

In [ ]:
injected_h5 = OUT / "loop_injected.h5"
with stg.Frame.open_copy(h5_path, injected_h5, overwrite=True, max_chunk_bytes=4096) as backed:
    result = backed.add_signal(
        path=stg.constant_path(
            backed.get_frequency(backed.fchans // 2 + 60),
            drift_rate=-0.4 * backed.unit_drift_rate,
        ),
        t_profile=stg.constant_t_profile(level=3),
        f_profile=stg.box_f_profile(width=5 * backed.df),
        auto_bounding=True,
    )
    print(result)

original = stg.Frame(waterfall=h5_path)
injected = stg.Frame(waterfall=injected_h5)
print("source unchanged:", np.allclose(original.data, frame.data))
print("output changed:", np.max(np.abs(injected.data - original.data)) > 0)